# Project Icarus - Interactive Dashboard

Use the widgets below to configure an interceptor engagement and visualize results in real time.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

from src.interceptors.config import InterceptorConfig
from src.guidance.law import GuidanceConfig, GuidanceLaw
from src.scenarios.target_factory import (
    BallisticScenario,
    FOBSScenario,
    HGVScenario,
    SuppressedScenario,
    SwarmScenario,
)
from src.scenarios.scenario import EngagementScenario
from src.sim.api import run_engagement

## 2. Widgets

In [ ]:
target_type_dropdown = widgets.Dropdown(
    options=["ballistic", "fobs", "hgp", "suppressed", "swarm"],
    value="ballistic",
    description="Target Type:",
)
interceptor_mass_slider = widgets.FloatSlider(
    value=1000.0, min=100.0, max=5000.0, step=100.0,
    description="Interceptor Mass (kg):",
)
kill_radius_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=10.0, step=0.1,
    description="Kill Radius (m):",
)
guidance_n_slider = widgets.FloatSlider(
    value=4.0, min=1.0, max=10.0, step=0.5,
    description="Guidance Gain N:",
)
n_trials_slider = widgets.IntSlider(
    value=20, min=5, max=100, step=5,
    description="MC Trials:",
)
run_button = widgets.Button(
    description="Run Engagement",
    button_style="success",
)
output_area = widgets.Output()

## 3. Dashboard Logic

In [ ]:
last_result = None

def make_target(target_type):
    if target_type == "ballistic":
        return BallisticScenario(r0=np.array([6371e3, 0.0, 0.0]), v0=np.array([0.0, 1000.0, 1000.0]))
    elif target_type == "fobs":
        return FOBSScenario.from_orbital_params(apoapsis_km=200.0, inclination_deg=30.0)
    elif target_type == "hgp":
        return HGVScenario.from_params(max_alt_km=80.0, lateral_range_km=2000.0)
    elif target_type == "suppressed":
        return SuppressedScenario(r0=np.array([6371e3, 0.0, 0.0]), v0=np.array([0.0, 800.0, 800.0]))
    elif target_type == "swarm":
        return SwarmScenario.from_params(n_payloads=3, spread_deg=2.0, range_km=500.0)
    else:
        return BallisticScenario()

def on_run_clicked(b):
    global last_result
    with output_area:
        clear_output(wait=True)
        interceptor = InterceptorConfig(
            name="Interactive",
            mass=interceptor_mass_slider.value,
            kill_radius=kill_radius_slider.value,
        )
        guidance_cfg = GuidanceConfig(
            terminal_n=guidance_n_slider.value,
            terminal_kill_radius=kill_radius_slider.value,
        )
        guidance = GuidanceLaw(config=guidance_cfg)
        target = make_target(target_type_dropdown.value)
        scenario = EngagementScenario(engagement_start=0.0, engagement_end=300.0)

        print(f"Running {n_trials_slider.value} trials for {target_type_dropdown.value} target...")
        result = run_engagement(interceptor, guidance, target, scenario, n_trials=n_trials_slider.value)
        last_result = result
        print(f"Nominal miss: {result.miss_distance:.2f} m")
        print(f"Kill: {result.kill_assessment}")
        if result.monte_carlo:
            print(f"MC kill probability: {result.monte_carlo.kill_probability:.2%}")
            print(f"Mean miss: {result.monte_carlo.mean_miss:.2f} m")
            print(f"Std miss: {result.monte_carlo.std_miss:.2f} m")

        fig = plt.figure(figsize=(12, 4))
        ax1 = fig.add_subplot(131, projection="3d")
        result.plot_3d(ax=ax1)
        ax1.set_title("Trajectory")

        ax2 = fig.add_subplot(132)
        result.plot_miss_distance_distribution(ax=ax2)
        ax2.set_title("Miss Distance Distribution")

        ax3 = fig.add_subplot(133)
        if result.monte_carlo:
            misses = result.monte_carlo.miss_distances
            kill_radius = kill_radius_slider.value
            kill_count = np.sum(np.array(misses) < kill_radius)
            ax3.bar(["Miss < Kill Radius", "Miss > Kill Radius"],
                     [kill_count, len(misses) - kill_count])
            ax3.set_title("Kill / No-Kill Count")
        plt.tight_layout()
        plt.show()

run_button.on_click(on_run_clicked)

## 4. Display Dashboard

In [ ]:
display(widgets.VBox([
    widgets.HBox([target_type_dropdown]),
    widgets.HBox([interceptor_mass_slider, kill_radius_slider]),
    widgets.HBox([guidance_n_slider, n_trials_slider]),
    run_button,
    output_area,
]))